# HierarchicalDet Phase 0 setup (Kaggle GPU notebook)

Scope: environment install, DENTEX dataset download + 3-tier verification, and a single-image forward-pass smoke test of the inference pipeline (no pretrained HierarchicalDet weights are publicly released, so this only proves the pipeline runs mechanically -- Phase 2 covers actual training/eval).

**Before running**: Settings > Accelerator > GPU T4x2 (or P100). Settings > Internet > On. Settings > Persistence > "Variables and files" (keeps the cloned repo / downloaded dataset across sessions -- the 11.8GB DENTEX download is slow, no reason to repeat it). Settings > Environment > "Pin to original environment".

**Every cell in this notebook is safe to re-run, and the whole notebook is safe to run top-to-bottom with "Run All" even after a session restart.** Kaggle's "pin to original environment" reverts to the base image's default packages on every fresh session -- anything pip-installed outside `/kaggle/working` does NOT survive a restart, only files under `/kaggle/working` do. So dependency installation is written to always re-run quickly (pip no-ops on already-satisfied packages) rather than being a one-time step, and the dataset download/extraction steps check for existing files first and skip the slow parts if already done.

Everything in this notebook -- the Pillow/legacy-PIL-constant issue, the pycocotools `_mask` extension, the dataset paths, the Swin-B checkpoint size mismatch -- has been verified end-to-end against a local Python 3.9/3.12 environment and real downloaded data before ever being run on Kaggle. If something still fails here, it's a genuinely new issue, not a repeat of one already chased down; paste the full traceback rather than just the last few lines, since some of the earlier bugs here only became clear from output further up (shape-mismatch warnings in particular are easy to miss in a truncated log).</cell id="cell-0">

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU in notebook Settings before proceeding.'

In [ ]:
# Clone (or update) the repo. Idempotent: safe to re-run after a restart --
# pulls latest instead of re-cloning if /kaggle/working/repo already exists.
REPO_URL = 'https://github.com/christopherh-88/HierarchicalDet.git'
%cd /kaggle/working
import os
if os.path.isdir('repo/.git'):
    %cd repo
    !git pull
else:
    !git clone $REPO_URL repo
    %cd repo

In [ ]:
# Install dependencies. Always re-run this on a fresh session -- see the
# top markdown cell for why pip installs don't persist across restarts here.
#
# Pillow note: earlier versions of this notebook tried to force-reinstall
# pillow<10 here to work around this old detectron2 snapshot's use of
# PIL.Image.LINEAR (removed in Pillow 10). That never reliably worked on
# Kaggle across three separate attempts (wrong interpreter, then PEP-668
# blocking installs silently, then a reinstall reporting success while doing
# nothing) -- and critically, even a notebook-level monkeypatch wouldn't have
# helped anyway, since demo.py runs as its own subprocess below and wouldn't
# inherit anything patched into this kernel's process. The real fix is now
# baked directly into the repo instead: detectron2/utils/env.py's
# _configure_libraries() (called on every `import detectron2`, regardless of
# entrypoint) restores the handful of legacy PIL.Image constants this
# codebase needs from the modern Resampling enum. Verified this works with
# Pillow 12.3 active and zero special handling needed here anymore.
import sys, subprocess

print('sys.executable:', sys.executable)

def pip_install(args):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + args
    try:
        subprocess.run(cmd + ['--break-system-packages'], check=True,
                        capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        if 'no such option' in (e.stderr or '').lower() or 'unrecognized arguments' in (e.stderr or '').lower():
            # older pip without PEP 668 support -- retry without the flag
            subprocess.run(cmd, check=True, capture_output=True, text=True)
        else:
            print(e.stdout)
            print(e.stderr)
            raise

# Exclude torch/torchvision/torchaudio (Kaggle ships a CUDA-matched build
# already) and pillow (no longer pinned -- see note above).
subprocess.run(
    "grep -v -i -E '^(torch|torchvision|torchaudio)$|^pillow' requirements.txt > /tmp/reqs_no_torch.txt",
    shell=True, check=True,
)
pip_install(['-r', '/tmp/reqs_no_torch.txt'])
print('Dependencies installed.')

In [ ]:
# Bundled pycocotools/ (custom categories_1/2/3 3-tier format) ships without
# its compiled _mask Cython extension, and repo-root imports always shadow
# the pip-installed pycocotools package. Copy the compiled extension for this
# interpreter/platform from the pip install into the bundled package.
import site, glob, shutil
site_pkgs = site.getsitepackages()[0]
mask_so = glob.glob(f'{site_pkgs}/pycocotools/_mask*.so')
assert mask_so, f'no compiled _mask extension found under {site_pkgs}/pycocotools'
shutil.copy(mask_so[0], 'pycocotools/')
print('copied', mask_so[0])

In [ ]:
# Verify the full import chain, same checks as local setup.sh
import detectron2
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from hierarchialdet import add_diffusiondet_config, DiffusionDetWithTTA
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper
from pycocotools.coco import COCO
import evaluator
print('All imports OK.')

## Download DENTEX and verify all three annotation tiers

Verified 2026-07-14 by directly inspecting the real dataset (not guessed --
`training_data.zip` is 10.9GB, so its internal file listing was pulled via an
HTTP range request against just the zip's central directory, and the three
annotation JSONs were fetched via their own byte ranges, without downloading
the whole archive):

- `training_data.zip` contains **four** subfolders, not three:
  `quadrant/` (693 images, `train_quadrant.json`, flat `category_id`, 4 quadrant classes),
  `quadrant_enumeration/` (634 images, `train_quadrant_enumeration.json`, `category_id_1/2`, 4+8 classes),
  `quadrant-enumeration-disease/` (705 images, `train_quadrant_enumeration_disease.json`, `category_id_1/2/3`, 4+8+4 classes -- diagnosis classes are Impacted/Caries/Periapical Lesion/Deep Caries, matching the paper), and
  `unlabelled/` (1572 images, no annotations at all -- not documented in either README, not used by the default pipeline).
- `validation_data.zip` contains only images, at `validation_data/quadrant_enumeration_disease/xrays/val_N.png` (50 of them). The matching annotations are **not** in the zip -- they're the separate top-level `DENTEX/validation_triple.json` (50 images, 182 annotations, confirmed to reference these exact filenames).
- **`test_data.zip` uses a completely different, incompatible format**: per-image LabelMe polygon JSON files (`disease/label/test_N.json`) with Turkish-language diagnosis strings baked into a single `label` field (e.g. `"1-çürük-15"` = quadrant-diagnosis-tooth), images at `disease/input/test_N.png`. This is **not** loadable by `register_coco_instances()` or `DiffusionDetDatasetMapper` as-is -- using the test split requires writing a LabelMe-to-COCO conversion script (parse polygons to bboxes, map the Turkish diagnosis vocabulary to `category_id_3`) that doesn't exist yet anywhere in this codebase. That conversion is Phase 1/2 scope; this notebook only reports the file counts and does not attempt it.
- Category ID *numbering* is not consistent across files (e.g. the diagnosis category order in `validation_triple.json` differs from `train_quadrant_enumeration_disease.json`) -- always read category names from each file's own `categories_N` list rather than assuming a fixed id-to-name mapping.

Also fixed as part of this verification: `detectron2/utils/visualizer.py` had a hardcoded, non-public path (`datasets/custom_triple/validation.json`) for loading category names during visualization, which crashed `demo.py`'s image-drawing step unconditionally. It's now controlled by a `DENTEX_CATEGORY_JSON` env var with a graceful fallback to numeric labels if unset (see the smoke-test cell below, which points it at the real `validation_triple.json`).</cell id="cell-6">

In [ ]:
# Download (skips already-downloaded files automatically -- huggingface_hub
# checks local_dir contents).
#
# Disk space note: Kaggle's /kaggle/working quota is limited (~20GB) and the
# raw downloaded zips alone are already 11.8GB. training_data.zip (10.9GB
# compressed) and test_data.zip (764MB) are NOT extracted here -- fully
# unpacking training_data.zip's images blew the disk quota entirely
# (confirmed on a real run: "OSError: [Errno 28] No space left on device").
# Phase 0 doesn't need those images extracted at all: the smoke test only
# needs one validation image, and the tier verification below reads stats
# directly out of the still-zipped archives via Python's zipfile (no
# extraction needed to list members or read a single small JSON entry).
# Only validation_data.zip (149.5MB, just images) is fully extracted, since
# that's small and is where the smoke-test image comes from.
pip_install(['huggingface_hub'])  # pip_install() defined in the dependency-install cell above

import os, shutil, zipfile
from huggingface_hub import snapshot_download

RAW_DIR = '/kaggle/working/dentex_raw'
EXTRACT_DIR = '/kaggle/working/dentex_extracted'
os.makedirs(EXTRACT_DIR, exist_ok=True)

dataset_dir = snapshot_download(
    repo_id='ibrahimhamamci/DENTEX',
    repo_type='dataset',
    local_dir=RAW_DIR,
)
print('downloaded to', dataset_dir)

TRAINING_ZIP = os.path.join(dataset_dir, 'DENTEX/training_data.zip')
VALIDATION_ZIP = os.path.join(dataset_dir, 'DENTEX/validation_data.zip')
TEST_ZIP = os.path.join(dataset_dir, 'DENTEX/test_data.zip')

# Clean up any partial extraction left over from an earlier run that hit the
# disk-space error, to reclaim space before anything else runs.
partial_training_dir = os.path.join(EXTRACT_DIR, 'training_data')
if os.path.isdir(partial_training_dir):
    print('removing partial training_data extraction from a previous run to reclaim disk space...')
    shutil.rmtree(partial_training_dir)
stale_marker = os.path.join(EXTRACT_DIR, '.DENTEX_training_data.zip.extracted')
if os.path.exists(stale_marker):
    os.remove(stale_marker)

# Only validation_data.zip gets fully extracted -- small, and it's where the
# smoke-test image comes from.
marker = os.path.join(EXTRACT_DIR, '.DENTEX_validation_data.zip.extracted')
if os.path.exists(marker):
    print('already extracted: validation_data.zip')
elif not os.path.exists(VALIDATION_ZIP):
    print('MISSING (not downloaded?):', VALIDATION_ZIP)
else:
    print('extracting', VALIDATION_ZIP, '...')
    with zipfile.ZipFile(VALIDATION_ZIP) as zf:
        zf.extractall(EXTRACT_DIR)
    open(marker, 'w').close()
    print('done: validation_data.zip')

print('training_data.zip and test_data.zip left unextracted (read via zipfile directly, see next cell).')
!df -h /kaggle/working

In [ ]:
# Verified real paths/member-names (see markdown cell above for how these
# were confirmed without downloading the full 10.9GB training_data.zip).
# Reads training/test tier stats directly out of the unextracted zips via
# zipfile (no disk space needed beyond the raw download) -- only the small
# validation_data.zip was actually extracted to disk, in the previous cell.
import json

TRAINING_TIER_MEMBERS = {
    'quadrant (train)': ('training_data/quadrant/train_quadrant.json', 'training_data/quadrant/xrays/'),
    'quadrant-enumeration (train)': ('training_data/quadrant_enumeration/train_quadrant_enumeration.json', 'training_data/quadrant_enumeration/xrays/'),
    'quadrant-enumeration-diagnosis (train)': ('training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json', 'training_data/quadrant-enumeration-disease/xrays/'),
}
UNLABELLED_PREFIX = 'training_data/unlabelled/xrays/'

VALIDATION_TRIPLE_PATH = os.path.join(dataset_dir, 'DENTEX', 'validation_triple.json')
VALIDATION_IMAGE_DIR = os.path.join(EXTRACT_DIR, 'validation_data', 'quadrant_enumeration_disease', 'xrays')

any_missing = False

if os.path.exists(TRAINING_ZIP):
    with zipfile.ZipFile(TRAINING_ZIP) as zf:
        names = zf.namelist()
        for tier, (json_member, img_prefix) in TRAINING_TIER_MEMBERS.items():
            if json_member not in names:
                print(tier, '-> MISSING member in zip:', json_member)
                any_missing = True
                continue
            d = json.loads(zf.read(json_member))
            n_images_on_disk_in_zip = sum(1 for n in names if n.startswith(img_prefix) and n.endswith('.png'))
            print(tier, '->', json_member, '(inside training_data.zip, not extracted)')
            print('  images (JSON):', len(d.get('images', [])), ' images (zip entries):', n_images_on_disk_in_zip, ' annotations:', len(d.get('annotations', [])))
            for k in d:
                if k.startswith('categor'):
                    print('  ', k, ':', [c['name'] for c in d[k]])
        n_unlabelled = sum(1 for n in names if n.startswith(UNLABELLED_PREFIX) and n.endswith('.png'))
        print('unlabelled xrays in zip (no annotations, not used by default pipeline):', n_unlabelled)
else:
    print('MISSING:', TRAINING_ZIP)
    any_missing = True

if os.path.exists(VALIDATION_TRIPLE_PATH):
    d = json.load(open(VALIDATION_TRIPLE_PATH))
    n_extracted = len([f for f in os.listdir(VALIDATION_IMAGE_DIR) if f.endswith('.png')]) if os.path.isdir(VALIDATION_IMAGE_DIR) else 0
    print('quadrant-enumeration-diagnosis (validation) ->', VALIDATION_TRIPLE_PATH)
    print('  images (JSON):', len(d.get('images', [])), ' images (extracted to disk):', n_extracted, ' annotations:', len(d.get('annotations', [])))
    for k in d:
        if k.startswith('categor'):
            print('  ', k, ':', [c['name'] for c in d[k]])
else:
    print('MISSING:', VALIDATION_TRIPLE_PATH)
    any_missing = True

print()
print('=== TEST SET: incompatible format, not COCO ===')
if os.path.exists(TEST_ZIP):
    with zipfile.ZipFile(TEST_ZIP) as zf:
        names = zf.namelist()
        n_test_images = sum(1 for n in names if n.startswith('disease/input/') and n.endswith('.png'))
        n_test_labels = sum(1 for n in names if n.startswith('disease/label/') and n.endswith('.json'))
    print(f'{n_test_images} test images, {n_test_labels} per-image LabelMe label files (Turkish diagnosis strings) -- inside test_data.zip, not extracted.')
    print('Not directly usable for evaluation -- needs a LabelMe-to-COCO conversion script (Phase 1/2 work, not done here).')
else:
    print('MISSING:', TEST_ZIP)

if any_missing:
    print()
    print('Some expected paths/members were missing -- inspect the zip contents directly, e.g.:')
    print("  zipfile.ZipFile(TRAINING_ZIP).namelist()[:50]")

## Single-image forward-pass smoke test

No pretrained HierarchicalDet weights exist publicly, so this uses the public
Swin-B ImageNet-22k backbone per `configs/diffdet.custom.swinbase.nonpretrain.yaml`,
with the detection head randomly initialized. The goal is only to prove the
model builds and runs a forward pass without crashing -- not to produce
meaningful detections (expect "detected 0 instances", confirmed below).

Verified locally end-to-end on 2026-07-14 (Python 3.12, CPU, a real
downloaded validation image) before this was ever run on Kaggle. This caught
a real bug along the way, now fixed: `configs/diffdet.custom.swinbase.nonpretrain.yaml`
had `SWIN.SIZE: L-22k` (Large, embed_dim=192) while its own `MODEL.WEIGHTS`
pointed at a Base-sized checkpoint filename (embed_dim=128) -- an internal
inconsistency in the original authors' own config (the file is literally
named "swinbase"). This caused 325 silent shape-mismatch warnings and meant
the backbone was effectively randomly initialized, not loading the
pretrained weights at all -- easy to miss since detectron2 logs shape
mismatches as warnings, not errors, and the earlier truncated terminal
output during debugging didn't surface them. Fixed by correcting `SIZE` to
`B-22k` to match the config's own weights path; confirmed zero shape
mismatches after the fix, with only the expected non-backbone components
(diffusion-process buffers, detection head) reported as unmatched, which is
correct since a full HierarchicalDet checkpoint is not what we're loading.</cell id="cell-9">

In [ ]:
# Download the Swin-B backbone (skipped if already present).
# IMPORTANT: keep the .pth extension -- detectron2's DetectionCheckpointer
# dispatches purely on file extension (detectron2/checkpoint/detection_checkpoint.py
# :_load_file): ".pkl" is parsed as a Caffe2-style pickle blob, which would
# silently mis-load a native torch.save() file. A non-".pkl"/".pyth" extension
# routes through the normal torch.load() + name-matching-heuristics path,
# which is what this raw Swin-Transformer release checkpoint needs.
# Confirmed 2026-07-14 (see markdown cell above): this checkpoint's keys do
# match hierarchialdet/swintransformer.py's naming, no conversion needed.
os.makedirs('models', exist_ok=True)
WEIGHTS_PATH = 'models/swin_base_patch4_window7_224_22k.pth'
if not os.path.exists(WEIGHTS_PATH):
    !wget -q -O $WEIGHTS_PATH https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_base_patch4_window7_224_22k.pth
print(WEIGHTS_PATH, os.path.getsize(WEIGHTS_PATH), 'bytes')

In [ ]:
# Use a known-real validation image (path verified above) and the real
# category names (DENTEX_CATEGORY_JSON -> validation_triple.json) instead of
# a placeholder or blind glob search.
sample_image = os.path.join(VALIDATION_IMAGE_DIR, 'val_0.png')
assert os.path.exists(sample_image), f'{sample_image} not found -- check the extraction/verification cells above ran successfully.'
print('using sample image:', sample_image)

import subprocess
env = dict(os.environ, DENTEX_CATEGORY_JSON=VALIDATION_TRIPLE_PATH)
result = subprocess.run([
    sys.executable, 'demo.py',
    '--config-file', 'configs/diffdet.custom.swinbase.nonpretrain.yaml',
    '--input', sample_image,
    '--output', '/kaggle/working/smoke_test_output.jpg',
    '--opts', 'MODEL.WEIGHTS', WEIGHTS_PATH, 'MODEL.DEVICE', 'cuda',
], env=env)
assert result.returncode == 0, f'demo.py exited with code {result.returncode}'
print('Smoke test complete -- see /kaggle/working/smoke_test_output.jpg')